In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import random
import numpy as np
import pandas as pd
import os
from omegaconf import OmegaConf

# SNPgen imports
from snpgen.data.loader import SplitDataset
from snpgen.utils import instantiate_from_config

# Evaluation module imports
from snpgen.evaluation import (
    # Pipeline
    train_models,
    # Results handling
    build_multiindex_df,
    save_cv_results,
    load_cv_results,
    check_results_exist,
    compute_gwas_prs_results,  # For GWAS PRS computation
    # Incremental CV training utilities
    verify_cv_indices,
    get_missing_trainers,
    merge_fold_results,
    # Analysis
    filter_model_list,
    # Plotting
    plot_metrics_with_ci,
    # Cross-validation utilities (supports binary classification)
    get_stratified_kfold,
)

In [ ]:
# ============================================================
# USER SETTINGS — Edit these before running
# ============================================================

# Path to your trained DDPM checkpoint directory.
# This directory must contain config.yaml and the generated .hdf5 files.
checkpoint_dir = '/path/to/snpgen/checkpoints/trait1/trait1_ddpm_emb128_small_white-31744535/'

# Choose the synthetic dataset types to evaluate
# Options: 'complete', 'augmented', 'reconstructed'
syn_dataset_types = ['complete', 'augmented', 'reconstructed']

# For 'reconstructed' mode only: which split to use for VAE reconstructions
# Options: 'train_val', 'test', 'full'
# Note: The reconstructed data will still be tested on the REAL test set
reconstruction_split = 'train_val'

# Cross-validation settings
n_folds = 5

# Models to train
# Valid names: 'xgboost', 'xgboost_balanced', 'catboost', 'prs'
default_models = ['xgboost', 'xgboost_balanced', 'catboost', 'prs']

# Optional: force retrain specific models even if results already exist
# Set to [] to skip. Use 'prs_gwas' to also force retrain GWAS PRS
force_retrain = []  # e.g., ['prs'] or ['prs', 'prs_gwas']

# Set to True to skip training on real data and only train on synthetic datasets
skip_real_training = False

In [ ]:
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
NUM_WORKERS = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
NUM_NODES = int(os.environ.get("SLURM_NNODES", 1))
ALLOCATED_GPUS_PER_NODE = int(os.environ.get("SLURM_GPUS_ON_NODE", 1))
SLURM_JOBID = os.environ.get("SLURM_JOB_ID", "local")

In [ ]:
num_gpus = torch.cuda.device_count()
print(f"{num_gpus} GPU(s) available")
print(f"Using {NUM_WORKERS} workers for the DataLoader")

# Load Datasets

In [ ]:
config_path = os.path.join(checkpoint_dir, 'config.yaml')

# =============================================================================
# Load Config and Real Dataset (once)
# =============================================================================

assert os.path.exists(config_path), f"config.yaml not found at {config_path}. This notebook requires a run with saved config.yaml."

print(f"Loading config from checkpoint directory: {config_path}")
config = OmegaConf.load(config_path)

h5_path = config['dataset_path']
print(f"Using dataset path from reloaded config: {h5_path}")
proj_name = os.path.basename(os.path.dirname(config['dataset_path'])).replace('ukb_', '')
print(f"Inferred project name: {proj_name}")

print(f"\nLoading Real Dataset from: {h5_path}")
if config.seed != seed:
    print(f"Overriding raw_dataset seed from {seed} to {config.seed}")
raw_dataset = instantiate_from_config(config.data.raw_dataset, file_path=h5_path, seed=config.seed, onehot=False, data_dtype=None, metadata=True)

# For reconstructed mode, auto-detect VAE checkpoint directory from DDPM config
vae_checkpoint_dir = None
if 'reconstructed' in syn_dataset_types:
    vae_ckpt_path = config.model.params.first_stage_config.params.get('ckpt_path', None)
    assert vae_ckpt_path is not None, (
        "'reconstructed' mode requires a VAE checkpoint path in the DDPM config "
        "(model.params.first_stage_config.params.ckpt_path)"
    )
    vae_checkpoint_dir = os.path.dirname(vae_ckpt_path)
    print(f"VAE checkpoint directory (from DDPM config): {vae_checkpoint_dir}")

# =============================================================================
# Determine ml_models directory (always binary classification)
# =============================================================================

ml_models_dirname = 'ml_models'
ml_models_path = os.path.join(os.path.dirname(h5_path), ml_models_dirname, 'cv')
os.makedirs(ml_models_path, exist_ok=True)

print(f"\nBinary classification task")
print(f"\nReal models output path: {ml_models_path}")

# =============================================================================
# Load Synthetic Datasets for each type
# =============================================================================

syn_datasets = {}

for syn_dataset_type in syn_dataset_types:
    print(f"\n{'='*60}")
    print(f"Loading {syn_dataset_type.upper()} Dataset")
    print(f"{'='*60}")
    
    assert syn_dataset_type in ['complete', 'augmented', 'reconstructed'], f"Invalid synthetic dataset type: {syn_dataset_type}"
    
    # Define synthetic dataset path based on type
    if syn_dataset_type == 'complete':
        h5_path_syn = os.path.join(checkpoint_dir, 'syn_complete_dataset.hdf5')
        ml_models_subdir = 'cv'
    elif syn_dataset_type == 'augmented':
        h5_path_syn = os.path.join(checkpoint_dir, 'syn_augmented_dataset.hdf5')
        ml_models_subdir = 'cv_augmented'
    elif syn_dataset_type == 'reconstructed':
        split_suffix = f"_{reconstruction_split}" if reconstruction_split else ""
        h5_path_syn = os.path.join(vae_checkpoint_dir, f'vae_reconstruction_dataset{split_suffix}.hdf5')
        ml_models_subdir = f'cv_reconstructed_{reconstruction_split}'
    
    # Check if dataset file exists
    if not os.path.exists(h5_path_syn):
        print(f"Skipping '{syn_dataset_type}' - Dataset file not found: {h5_path_syn}")
        continue
    
    print(f"Path: {h5_path_syn}")
    
    # Load synthetic dataset
    raw_dataset_syn = instantiate_from_config(
        config.data.raw_dataset,
        file_path=h5_path_syn,
        seed=config.seed,
        onehot=False,
        data_dtype=None,
        metadata=True,
        x_key='syn_samples',
        y_key='targets',
    )
    
    # Determine output path
    if syn_dataset_type == 'reconstructed':
        ml_models_path_syn = os.path.join(vae_checkpoint_dir, ml_models_dirname, ml_models_subdir)
    else:
        ml_models_path_syn = os.path.join(checkpoint_dir, 'ml_models', ml_models_subdir)
    os.makedirs(ml_models_path_syn, exist_ok=True)
    
    print(f"  Output path: {ml_models_path_syn}")
    
    syn_datasets[syn_dataset_type] = {
        'h5_path': h5_path_syn,
        'dataset': raw_dataset_syn,
        'ml_models_path': ml_models_path_syn,
    }

print(f"\n{'='*60}")
print(f"Successfully loaded {len(syn_datasets)} synthetic dataset(s): {list(syn_datasets.keys())}")
print(f"{'='*60}")

# CV Pipeline

<div style="text-align: center;">
    <div style="background-color: #FFA500; color: black; font-weight: bold; padding: 10px; border-radius: 5px; width: 60%">
        FULL DATASET (real)
    </div>
    <div style="display: flex; justify-content: center; margin-top: 10px;">
        <div style="background-color: #4CAF50; color: black; font-weight: bold; padding: 10px; border-radius: 5px; width: 50%;">
            TRAINING SET (real) (used to train the generative model)
        </div>
        <div style="background-color: #ff6666; color: black; font-weight: bold; padding: 10px; border-radius: 5px; width: 10%; margin-left: 5px;">
            TEST SET (real)
        </div>
    </div>
</div>
 &nbsp;
<div style="text-align: center;">
    <div style="background-color: #3399ff; color: black; font-weight: bold; padding: 10px; border-radius: 5px; width: 60%">
        FULL DATASET (syn)
    </div>
    <div style="display: flex; justify-content: center; margin-top: 10px;">
        <div style="background-color: #e0b3ff; color: black; font-weight: bold; padding: 10px; border-radius: 5px; width: 50%;">
            TRAINING SET (syn)
        </div>
        <div style="color: black; font-weight: bold; padding: 10px; border-radius: 5px; width: 10%; margin-left: 5px; background: linear-gradient(20deg, transparent 45%, red 45%, red 55%, transparent 55%); background-color: #f5f5f5;">
            TEST SET (unused)
        </div>
    </div>
</div>

The splits in each iteration will be the following (example with 5 folds):

<div style="display: flex; justify-content: space-around; gap: 50px;">
    
<div>
    <h4 style="text-align: center;"></h4>

| &nbsp; | 
|---------|
| **Iteration 1** |
| **Iteration 2** | 
| **Iteration 3** |
| **Iteration 4** | 
| **Iteration 5** |

</div>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<div>
    <h4 style="text-align: center;"><span style="color:#4CAF50">TRAINING SET (real)</span></h4>

| Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
|--------|--------|--------|--------|--------|
| **Unused** | Train  | Train  | Train  | Train  |
| Train   | **Unused** | Train  | Train  | Train  |
| Train   | Train   | **Unused** | Train  | Train  |
| Train   | Train   | Train  | **Unused** | Unused  |
| Train   | Train   | Train  | Train  | **Test** |

</div>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<div>
    <h4 style="text-align: center;"><span style="color:#e0b3ff">TRAINING SET (syn)</span></h4>

| Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
|--------|--------|--------|--------|--------|
| **Unused** | Train  | Train  | Train  | Train  |
| Train   | **Unused** | Train  | Train  | Train  |
| Train   | Train   | **Unused** | Train  | Train  |
| Train   | Train   | Train  | **Unused** | Unused  |
| Train   | Train   | Train  | Train  | **Test** |

</div>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<div>
    <h4 style="text-align: center;"><span style="color:#ff6666">TEST SET (real)</span></h4>

| Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
|--------|--------|--------|--------|--------|
| **Test** | Unused  | Unused  | Unused  | Unused  |
| Unused   | **Test** | Unused  | Unused  | Unused  |
| Unused   | Unused   | **Test** | Unused  | Unused  |
| Unused   | Unused   | Unused  | **Test** | Unused  |
| Unused   | Unused   | Unused  | Unused  | **Test** |

</div>

</div>

So, for instance, in the first iteration the ML models will be trained on Fold2+Fold3+Fold4+Fold5 of the Real Training Set and will be also trained on Fold2+Fold3+Fold4+Fold5 of the Syn Training Set. Then, in both cases they will be tested on Fold1 of the Real Test Set. \
This is not the usual KFold configuration, but it is a reasonable way to have a confidence interval for the ML models in both the Real and Syn scenarios without using as a test set parts of the data used to train the generative model.

In [ ]:
# Get real data splits
real_split_data, real_split_labels, real_split_metadata = raw_dataset.get_split('train_val', metadata=True)
test_split_data, test_split_labels, test_split_metadata = raw_dataset.get_split('test', metadata=True)

# Stratified k-fold for binary classification
skf_real = get_stratified_kfold(real_split_labels, n_splits=n_folds, shuffle=True, random_state=seed)
skf_test = get_stratified_kfold(test_split_labels, n_splits=n_folds, shuffle=True, random_state=seed)

# Check if GWAS betas are available for PRS computation
gwas_betas = raw_dataset.metadata.get('betas', None) if raw_dataset.metadata else None
has_gwas_betas = gwas_betas is not None
if has_gwas_betas:
    print(f"GWAS betas available: shape {gwas_betas.shape}")
else:
    print("No GWAS betas found in metadata - skipping GWAS PRS")

# =============================================================================
# REAL DATA TRAINING (incremental - only trains missing models)
# =============================================================================

# Load existing results if available
existing_results_real = None
if check_results_exist(ml_models_path):
    existing_results_real = load_cv_results(ml_models_path)
    print(f"\n{'='*60}")
    print("REAL DATA: Loaded existing results")
    print(f"  Path: {ml_models_path}")
    first_fold = list(existing_results_real.keys())[0]
    existing_model_names = list(existing_results_real[first_fold].get('models', {}).keys())
    print(f"  Existing models in {first_fold}: {existing_model_names}")
    print(f"{'='*60}")
else:
    print(f"\n{'='*60}")
    print("REAL DATA: No existing results found. Training all models...")
    print(f"{'='*60}")

_cv_results_real = existing_results_real if existing_results_real is not None else {}
any_real_trained = False

if not skip_real_training:
    for i, ((train_index, _), (_, test_index)) in enumerate(zip(
            skf_real.split(real_split_data, real_split_labels),
            skf_test.split(test_split_data, test_split_labels)
        )):
        
        fold_key = f'fold_{i}'
        
        X_train, y_train = real_split_data[train_index], real_split_labels[train_index]
        X_test, y_test = test_split_data[test_index], test_split_labels[test_index]

        fold_metadata = {'train_index': train_index, 'test_index': test_index}

        # Verify indices match if results exist
        if existing_results_real is not None:
            verify_cv_indices(_cv_results_real, fold_key, train_index, test_index)
        
        # Determine which models are missing
        missing_trainers = get_missing_trainers(_cv_results_real, fold_key, default_models, force_retrain=force_retrain)
        need_gwas = has_gwas_betas and (
            not any(k.startswith('prs gwas') for k in _cv_results_real.get(fold_key, {}).get('models', {}))
            or (force_retrain and 'prs_gwas' in force_retrain)
        )
        
        if not missing_trainers and not need_gwas:
            print(f"\n*** Fold {i}: All models already exist, skipping ***")
            continue
        
        any_real_trained = True
        print(f"\n*** Fold {i}: ***")
        
        if missing_trainers:
            print(f"  Missing trainers: {missing_trainers}")
            print(f"\n++ Training on Real (models: {missing_trainers}) ++")
            real_results = train_models(
                X_train, y_train, X_test, y_test,
                metrics_dict=None,
                save_path=os.path.join(ml_models_path, fold_key),
                on_gpu=True, seed=seed,
                models=missing_trainers,
            )
            merge_fold_results(_cv_results_real, real_results, fold_key, fold_metadata)
            save_cv_results(_cv_results_real, ml_models_path)  # Save after each fold to ensure progress is not lost
        
        # Add GWAS PRS if betas available and not already present
        if need_gwas:
            print("\n++ Computing GWAS PRS (fixed betas) ++")
            gwas_prs_results = compute_gwas_prs_results(
                gwas_betas, X_test, y_test, model_name='prs gwas', verbose=True,
            )
            merge_fold_results(_cv_results_real, gwas_prs_results, fold_key, fold_metadata)

    if any_real_trained:
        print("\nSaving updated real data results...")
        save_cv_results(_cv_results_real, ml_models_path)
        print(f"Real data results saved to: {ml_models_path}")
    else:
        print("\nAll real data models already trained. No changes needed.")

# =============================================================================
# SYNTHETIC DATA TRAINING (incremental - only trains missing models)
# =============================================================================

all_cv_results_syn = {}

for syn_dataset_type, syn_data in syn_datasets.items():
    print(f"\n{'='*60}")
    print(f"{syn_dataset_type.upper()} DATA: Starting CV Pipeline")
    print(f"{'='*60}")
    
    raw_dataset_syn = syn_data['dataset']
    ml_models_path_syn = syn_data['ml_models_path']
    
    # Load existing results if available
    existing_results_syn = None
    if check_results_exist(ml_models_path_syn):
        existing_results_syn = load_cv_results(ml_models_path_syn)
        first_fold = list(existing_results_syn.keys())[0]
        existing_model_names = list(existing_results_syn[first_fold].get('models', {}).keys())
        print(f"  Loaded existing results from: {ml_models_path_syn}")
        print(f"  Existing models in {first_fold}: {existing_model_names}")
    else:
        print(f"  No existing results. Training all models...")
    
    _cv_results_syn = existing_results_syn if existing_results_syn is not None else {}
    
    # Get synthetic split
    if syn_dataset_type == 'reconstructed':
        syn_split_data, syn_split_labels, syn_split_metadata = raw_dataset_syn.get_split('full', metadata=True)
        print(f"  Using 'full' split for reconstructed data (already contains {reconstruction_split} split)")
    else:
        syn_split_data, syn_split_labels, syn_split_metadata = raw_dataset_syn.get_split('train_val', metadata=True)
    
    skf_syn = get_stratified_kfold(syn_split_labels, n_splits=n_folds, shuffle=True, random_state=seed)
    
    # Check if any training is needed across all folds
    any_syn_trained = False
    
    for i, ((train_syn_index, _), (_, test_index)) in enumerate(zip(
            skf_syn.split(syn_split_data, syn_split_labels),
            skf_test.split(test_split_data, test_split_labels)
        )):
        
        fold_key = f'fold_{i}'
        
        X_train_syn, y_train_syn = syn_split_data[train_syn_index], syn_split_labels[train_syn_index]
        X_test, y_test = test_split_data[test_index], test_split_labels[test_index]

        fold_metadata = {'train_index': train_syn_index, 'test_index': test_index}

        # Verify indices match if results exist
        if existing_results_syn is not None:
            verify_cv_indices(_cv_results_syn, fold_key, train_syn_index, test_index)
        
        # Determine which models are missing
        missing_trainers = get_missing_trainers(_cv_results_syn, fold_key, default_models, force_retrain=force_retrain)
        
        if not missing_trainers:
            print(f"\n*** Fold {i}: All models already exist, skipping ***")
            continue
        
        any_syn_trained = True
        print(f"\n*** Fold {i}: ***")
        print(f"  Missing trainers: {missing_trainers}")
        
        print(f"\n++ Training on {syn_dataset_type.capitalize()} (models: {missing_trainers}) ++")
        syn_results = train_models(
            X_train_syn, y_train_syn, X_test, y_test,
            metrics_dict=None,
            save_path=os.path.join(ml_models_path_syn, fold_key),
            on_gpu=True, seed=seed,
            models=missing_trainers,
        )
        merge_fold_results(_cv_results_syn, syn_results, fold_key, fold_metadata)
        save_cv_results(_cv_results_syn, ml_models_path_syn)  # Save after each fold to ensure progress is not lost

    if any_syn_trained:
        print(f"\nSaving updated {syn_dataset_type} data results...")
        save_cv_results(_cv_results_syn, ml_models_path_syn)
        print(f"{syn_dataset_type.capitalize()} data results saved to: {ml_models_path_syn}")
    else:
        print(f"\nAll {syn_dataset_type} models already trained. No changes needed.")
    
    all_cv_results_syn[syn_dataset_type] = _cv_results_syn

print("\n\n" + "="*60)
print("Cross-Validation Training Complete")
print("="*60)
print(f"\nReal data results: {ml_models_path}")
for syn_type, syn_data in syn_datasets.items():
    print(f"{syn_type.capitalize()} data results: {syn_data['ml_models_path']}")

# Plots

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

In [ ]:
# =============================================================================
# PLOTTING with AUTO-DETECTION
# =============================================================================
# The plot_metrics_with_ci function automatically detects whether the data
# is from a classification (binary) task and selects appropriate metrics:
#   - Classification (binary): ['balanced_accuracy', 'roc_auc']
#
# You can still override this behavior by specifying metrics_to_plot explicitly.
# =============================================================================

load_scaled_results = False

# Models to include in the plot (leave empty for all models)
# NOTE: GWAS PRS is shown as horizontal reference line, not as bars
models_to_keep = [
    'xgboost',
    'xgboost_balanced',
    'prs univariate scaled (threshold 0.5)',
]

base_paths = {
    '/path/to/snpgen/data/ukb_<trait>/ml_models/cv': {
        'pretty_name': 'Real',
        'add_prs': False,
        'add_univariate_prs': True,
        'color': '#D4EAF2'
    },
    '/path/to/your/vae/checkpoint/ml_models/cv_reconstructed_train_val/': {
        'pretty_name': 'Reconstructed',
        'add_prs': False,
        'add_univariate_prs': True,
        'color': '#4A90E2'
    },
    '/path/to/your/ddpm/checkpoint/ml_models/cv': {
        'pretty_name': 'Syn',
        'add_univariate_prs': True,
        'color': '#FB9270'
    },
    '/path/to/your/ddpm/checkpoint/ml_models/cv_augmented': {
        'pretty_name': 'Syn Augmented',
        'add_prs': False,
        'add_univariate_prs': True,
        'color': '#9C2A20'
    },
}

# Optional: Override model display names
model_pretty_names = {
    'random_forest': 'Random\nForest',
    'catboost': 'CatBoost',
    'xgboost': 'XGBoost',
    'xgboost_balanced': 'XGBoost\n(Balanced)',
    'knn': 'kNN',
    'prs scaled (threshold 0.5)': 'PRS\n(scaled [0-1])',
    'prs univariate scaled (threshold 0.5)': 'PRS Univ\n(scaled [0-1])',
    'prs univariate': 'PRS Univ',
}

filename = 'results_scaled.pkl' if load_scaled_results else 'results.pkl'

results_df = []
colors = []

gwas_prs_df = None  # To store GWAS PRS results from real data

for path, properties in base_paths.items():
    r = load_cv_results(path, filename)
    df = build_multiindex_df(r)
    
    if 'real' in properties['pretty_name'].lower():
        gwas_prs_df = df.copy()  # Update GWAS PRS df from real data if available
    
    # Filter models if specified
    if len(models_to_keep) > 0:
        available_models = df.index.get_level_values('model').unique().to_list()
        models_in_df = [m for m in models_to_keep if m in available_models]
        if models_in_df:
            df = df.loc[:, models_in_df, :]
        
    model_keys = filter_model_list(
        df.index.get_level_values('model').unique().to_list(),
        include_prs=properties.get('add_prs', False),
        include_prs_univariate=properties.get('add_univariate_prs', False),
        prs_to_include=['prs scaled (threshold 0.5)'],
        prs_univariate_to_include=['prs univariate scaled (threshold 0.5)', 'prs univariate'],
    )
    df = df.loc[:, model_keys, :]     
        
    colors.append(properties.get('color', 'black'))
    df = df.assign(name=properties['pretty_name'])
    results_df.append(df)

combined_df = pd.concat(results_df)
combined_df = combined_df.set_index('name', append=True).reorder_levels(['name', 'fold', 'model'])

# Extract trait name from Real data path for plot title
real_path = [path for path, props in base_paths.items() if props.get('pretty_name') == 'Real'][0]
if 'ukb_' in real_path:
    trait_name = real_path.split('ukb_')[1].split('/')[0].replace('_', ' ').title()
else:
    trait_name = None

# AUTO-DETECTION: No need to specify metrics_to_plot or metric_pretty_names
# The function will detect the task type and use appropriate defaults
fig = plot_metrics_with_ci(
    combined_df,
    model_pretty_names=model_pretty_names,
    colors=colors,
    confidence=0.95,
    joint=True,
    orientation='vertical',
    base_height=4,
    base_width=8,
    show_prs_baseline=True,  # Show univariate PRS as horizontal reference line
    show_gwas_prs_baseline=True,  # Show GWAS PRS as horizontal reference line
    gwas_prs_df=gwas_prs_df,  # Pass full df - function selects the right model
    title=trait_name,  # Add trait name as title
    # task_type='auto',  # Default - auto-detects from data
    # metrics_to_plot=['roc_auc'], 
    # output_path=f"/path/to/output/{trait_name}/metrics.png",  # Set to a path to save the figure and .csv
)